# Flow-direction analysis: SFD vs. two-neighbour vs. MFD routing

This notebook compares three `gospl` landscape-evolution experiments that differ **only** in how water is routed downhill: single-flow direction (SFD, one downstream receiver), a two-neighbour scheme, and multiple-flow direction (MFD, flow shared among all lower neighbours, weighted by slope). All three runs use the same rotationally-symmetric "sombrero" surface (100 x 100 km, 200 m resolution), forced by 1 m/yr uniform precipitation over 100,000 years with stream-power incision $E = K A^m S^n$ ($K = 4\times10^{-6}$) and hillslope diffusion $\partial z/\partial t = \kappa \nabla^2 z$ ($\kappa = 0.1$ m$^2$/yr).

After running the simulations externally, this notebook remaps the unstructured outputs onto a regular grid, exports them as netCDF, and visualises how the routing choice changes the resulting topography and flow discharge. The closing discussion explains why SFD drainage patterns are mesh-dependent artefacts while MFD reduces grid-resolution sensitivity.

## Setup

We import the scientific stack used throughout: `numpy`/`xarray` for array and netCDF handling, `matplotlib` for plotting, and the local `scripts.mapOutputs` helper (`mout`) that remaps `gospl`'s unstructured Voronoi outputs onto a regular grid and writes netCDF files.

Import required Python packages for this notebook.


In [ ]:
import os
import numpy as np
import xarray as xr

from scripts import mapOutputs as mout

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
%matplotlib inline

### Optional: tectonic forcing file

The commented block below shows how the vertical-displacement file (`tec.npz`) used by the experiments was built: a uniform uplift/subsidence rate of $-0.05$ m/yr (negative = subsidence) is assigned to every mesh vertex. It is provided for reference and is not needed to re-run the analysis.

> In goSPL, it is possible to use different flow-routing algorithms by specifying the number of directions appropriately weighted by the slope that rivers could potentially take when moving downhill.

**In this example, we run a series of three experiments in which the flow-routing calculations are based on one (SFD), two, and multiple (MFD) flow direction approaches.**


#### Mesh creation

Here, the initial surface consists of a rotationally symmetric surface  composed of valleys and ridges. The triangulated grid of 100 by 100 km is built with a resolution of 200 m. The three experiments with varying water-routing directions are run for 100,000 years with a Δt of 1000 years under a 1 m/yr uniform precipitation. 

In addition to stream incision (bedrock erodibility K set to 4.e-6), hillslope processes are also accounted for using a diffusion coefficient D of 0.1 m2/yr.

The initial mesh (`sombrero.npz`) has already been generated and is available in the the `data` folder. Below we provide the code needed to regenerate it if necessary.

In [ ]:
make_mesh = True
if make_mesh:
    import shutil
    import meshio
    import uxarray as uxr
    from scripts import umeshFcts as ufcts
    from gospl.mesher.meshfunc import VoroBuild

    output_path = "sombrero" 
    shutil.rmtree(output_path, ignore_errors=True)
    if not os.path.exists(output_path):
        os.makedirs(output_path)

    dx = 200 # desired resolution
    nx = 500 # desired number of nodes along the x-axis
    ny = 500 # desired number of nodes along the y-axis

    xcoords = np.arange(nx)*float(dx) 
    ycoords = np.arange(ny)*float(dx) 

    x1 = np.arange(-6.25, 6.25, 0.025)
    y1 = np.arange(-6.25, 6.25, 0.025)

    X1, Y1 = np.meshgrid(x1, y1)

    coords1 = np.vstack([X1.ravel(), Y1.ravel()])

    xx = coords1[0,:]
    yy = coords1[1,:]

    radius  = np.sqrt((xx**2 + yy**2))
    theta   = np.arctan2(yy,xx)

    height  = np.exp(-0.025*(xx**2 + yy**2)**2) + 0.25 * (0.15*radius)**4  * np.cos(10.0*theta)**2 
    height  += 0.5 * (1.0-0.3*radius)

    height = np.multiply(height,333.)+75.
    noise = np.random.normal(0, 1., height.shape)

    ids = np.where(height>100)[0]
    height += noise*0.75
    height[ids] += noise[ids]*2.0
    
    ds = xr.Dataset({
        'elev': xr.DataArray(
                    data   = height.reshape(X1.shape),
                    dims   = ['y','x'],
                    coords = {'x': xcoords,'y': ycoords},
                    ),
            }
        )
    ds['cellwidth'] = (['y','x'],dx*np.ones( (ny, nx)))
        
    # Build your planar mesh
    ufcts.planarMesh(ds,output_path,fvtk='planar.vtk',fumpas=True,voro=True)

    # Loading unstructured grid file
    var_name = 'data'
    ufile = output_path + '/base2D.nc' 
    mapds = xr.open_dataset(ufile)

    # Perform the interpolation (bilinear) 
    ufcts.inter2UGRID(ds[['elev']],mapds,output_path,var_name,type='face',latlon=False)
    data_ds = xr.open_dataset(output_path + '/' + var_name + '.nc')

    # --- Nodes (vertices in MPAS = dual mesh nodes) ---
    n_nodes = mapds.dims['nCells']
    ucoords = np.zeros((n_nodes, 3))
    ucoords[:, 0] = mapds['xCell'].values
    ucoords[:, 1] = mapds['yCell'].values
    ucoords[:, 2] = mapds['zCell'].values

    # --- Faces (cells in MPAS = primal mesh faces) ---
    ufaces = mapds['cellsOnVertex'].values - 1  

    print(f"Number of nodes: {len(ucoords)} | Number of faces: {len(ufaces)}")

    # --- Edge lengths (dcEdge = distance between cell centres across each edge) ---
    dcEdge = mapds['dcEdge'].values  # in metres
    edge_min  = np.round(dcEdge.min()  / 1000., 2)
    edge_max  = np.round(dcEdge.max()  / 1000., 2)
    edge_mean = np.round(dcEdge.mean() / 1000., 2)
    print(f"Edge range (km): min {edge_min} | max {edge_max} | mean {edge_mean}")

    mesh = meshio.read(output_path+'/planar.vtk')
    vertex = mesh.points
    cells = mesh.cells_dict['triangle']
    Umesh = VoroBuild()
    Umesh.initVoronoi(vertex, cells)
    Uarea = Umesh.control_volumes
    print('Cell area (km2): ',Uarea.min()*1.e-6,Uarea.max()*1.e-6)

    meshname = output_path+"/sombrero"
    np.savez_compressed(meshname, v=vertex, c=cells, 
                    z=data_ds.elev.data,
                    )

Import numpy as np


In [ ]:
# output_path = "data" 
# meshname = output_path+"/sombrero.npz"
# v = np.load(meshname)['v']
# uv = np.ones((len(v),2))*-0.05
# np.savez_compressed('data/tec.npz', uv=uv)

You will find a series of 3 goSPL input files:
- input-sfd.yml
- input-2ngb.yml
- input-mfd.yml

These files use the same elevation and forcing conditions, the only difference being the number of downstream nodes used when computing flow directions.

## Running the simulations

First activate the conda environment:

```bash
conda activate gospl
```

To run the simulation, you will need to do the following in a terminal:

```bash
mpirun -np X gospl -i input-escarpment.yml 
```

where X is the number of processors to use (for example 5), and XX is the name of the input file.